In [1]:
##############
## INITIATE ##
##############

## Imports
import os
import torch
from torch.utils.data import DataLoader, random_split

## Import modules
from f_dataset_augmented import ImageDataset
from f_model_resnet import ResNet18FeatureExtractor
from f_model_unet import UNetFeatureExtractor
from f_model_transunet import UNetTransformerFeatureExtractor
from f_model_transformer import VisionTransformerEncoder
from f_train_contrastive import train
# from f_train_contrastive_simCLR import train

#
###

In [2]:
###########################
## SCRIPT CONFIGURATIONS ##
###########################

## Set device
device = 'mps'

## Define paths and parameters
dataset_path = '/Volumes/Elements/BDI/projects/GSG/outputs/o1_extracted_glands/tcga-oct/'
# dataset_path = '/Users/user/Documents/projects/GlandSeg/outputs/V2_0_0_20250120/o1_extracted_glands/debug'
# dataset_path = '/Users/user/Documents/projects/GlandSeg/outputs/V2_0_0_20250120/o1_extracted_glands/tcga_ffpes_V1'
RESIZE = 224
batch_size = 128
val_split = 0.2
test_split = 0.1
FILE_FORMAT = '_raw.png'

#
###

In [3]:
#########################
## LOAD AND SPLIT DATA ##
#########################

## Load dataset
dataset = ImageDataset(folder_path=dataset_path, augment=False, resize=RESIZE, file_format=FILE_FORMAT)

## Split dataset into train, validation, and test
train_size = int((1 - val_split - test_split) * len(dataset))
val_size = int(val_split * len(dataset))
test_size = len(dataset) - train_size - val_size
#
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

## Create data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

#
###

In [4]:
########################
## DEFINE THE ENCODER ##
########################

## Initialize the encoder model
# model = UNetFeatureExtractor(n_classes=3).to(device)
# model = UNetTransformerFeatureExtractor(n_classes=3).to(device)
model = ResNet18FeatureExtractor().to(device)
# model = VisionTransformerEncoder(dim=512).to(device)

## Path to the saved weights file
weights_path = 'models/model_resnet_tcgaoct_raw_FTL_3.pth'
# weights_path = 'models/model_resnet_tcga_raw_FTL_0.pth'

if os.path.exists(weights_path):
    ## Load the state dictionary (weights)
    state_dict = torch.load(weights_path, map_location=device)
    ## Load the weights into the model
    model.load_state_dict(state_dict)
    print('Loaded weights')

## Save the fine-tuned model
torch.save(model.state_dict(), weights_path)

#
###

/opt/anaconda3/envs/pytorch/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/var/folders/mf/6vvt1nwx5zz1gr1nj3hnqfzr0000gp/T/ipykernel_87572/3818221697.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://g

Loaded weights


In [5]:
################
## TRAIN LOOP ##
################

## Parameters
embedding_dim = 512
projection_dim = 128
num_epochs = 10
device = 'mps'
pt_encoder = 'models/model_resnet_tcgaoct_raw_FTL_10.pth'
# temperature = 0.07
margin = 0.75
alpha = 0.0

## Pretrain encoder using the contrastive learning approach
# encoder, train_loss, test_loss = train(model, train_loader, val_loader, embedding_dim, projection_dim, num_epochs, device, temperature)
encoder, train_loss, test_loss = train(model, train_loader, val_loader, embedding_dim, projection_dim, num_epochs, device, margin, alpha)

## Save the fine-tuned model
torch.save(encoder.state_dict(), pt_encoder)

#
###

Epoch [1/10], Batch [1000/1376], Avg Loss: 0.1286
Epoch 1, Train Loss: 0.1246, Test Loss: 0.4190
Epoch [2/10], Batch [1000/1376], Avg Loss: 0.0993
Epoch 2, Train Loss: 0.0970, Test Loss: 0.3855
Epoch [3/10], Batch [1000/1376], Avg Loss: 0.0833
Epoch 3, Train Loss: 0.0840, Test Loss: 0.4785
Epoch [4/10], Batch [1000/1376], Avg Loss: 0.0788
Epoch 4, Train Loss: 0.0779, Test Loss: 0.4622
Epoch [5/10], Batch [1000/1376], Avg Loss: 0.0731
Epoch 5, Train Loss: 0.0725, Test Loss: 0.4205
Epoch [6/10], Batch [1000/1376], Avg Loss: 0.0677
Epoch 6, Train Loss: 0.0672, Test Loss: 0.4389
Epoch [7/10], Batch [1000/1376], Avg Loss: 0.0691
Epoch 7, Train Loss: 0.0672, Test Loss: 0.5524
Epoch [8/10], Batch [1000/1376], Avg Loss: 0.0645
Epoch 8, Train Loss: 0.0647, Test Loss: 0.3679
Epoch [9/10], Batch [1000/1376], Avg Loss: 0.0606
Epoch 9, Train Loss: 0.0607, Test Loss: 0.4865
Epoch [10/10], Batch [1000/1376], Avg Loss: 0.0585
Epoch 10, Train Loss: 0.0587, Test Loss: 0.4998
